## Imports

In [0]:
from pyspark.sql import SparkSession
import pandas as pd

## Get the needed Parameters

In [0]:
run_date = dbutils.widgets.get("run_date")

Businessdate = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast({run_date!r} as date) is not null then try_cast({run_date!r} as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `p-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast({run_date!r} as date),cast(getdate() as date))
                    """).first()[0]

print(Businessdate)

## Get the needed Source Tables

In [0]:
Portfolio = spark.sql(f"""                   
                            SELECT 31    AS unit_code_profits
                                 , 'DMW' AS Prefix
                            UNION          
                            SELECT 30    AS unit_code_profits
                                 , 'DMR' AS Prefix
                            UNION          
                            SELECT 3031  AS unit_code_profits
                                 , 'DMD' AS Prefix
                        """)

Portfolio.createOrReplaceTempView("Portfolio")
display(Portfolio)

In [0]:
loan_details = spark.sql(f"""                    
                            select distinct agreement_number, account_number, prft_acc_unit, prft_acc_sn, prft_acc_type
                            from `p-cdh`.landed.profits_loan_details
                            where BusinessDate = '{Businessdate}'
                        """)

loan_details.createOrReplaceTempView("loan_details")
# display(loan_details)

In [0]:
loan_balance = spark.sql(f"""                    
                        select distinct account_number
                        from `p-cdh`.landed.profits_loan_balance
                        where BusinessDate = '{Businessdate}'
                    """)

loan_balance.createOrReplaceTempView("loan_balance")
display(loan_balance)

In [0]:
loan_relation = spark.sql(f"""                    
                            select distinct FK_ACCOUNT
                            from `p-cdh`.landed.profits_loan_relation
                            where BusinessDate = '{Businessdate}'
                        """)

loan_relation.createOrReplaceTempView("loan_relation")
# display(loan_relation)

In [0]:
loan_request = spark.sql(f"""                    
                            select distinct ACCOUNT_NUMBER, unit_code
                            from `p-cdh`.landed.profits_loan_request
                            where BusinessDate = '{Businessdate}'
                        """)

loan_request.createOrReplaceTempView("loan_request")
# display(loan_request)

In [0]:
TUN = spark.sql(f"""                    
                    select distinct ACCOUNT_NUMBER
                    from `p-cdh`.landed.profits_TUN
                    where BusinessDate = '{Businessdate}'
                """)

TUN.createOrReplaceTempView("TUN")
display(TUN)

In [0]:
write_off_account = spark.sql(f"""                    
                                select distinct ACCOUNT_NUMBER
                                from `p-cdh`.landed.profits_write_off_account
                                where BusinessDate = '{Businessdate}'
                            """)

write_off_account.createOrReplaceTempView("write_off_account")
# display(write_off_account)

In [0]:
agreement = spark.sql(f"""                    
                        select distinct Agreement_Number, unit_code
                        from `p-cdh`.landed.profits_agreement
                        where BusinessDate = '{Businessdate}'
                    """)

agreement.createOrReplaceTempView("agreement")
# display(agreement)

In [0]:
agreement_relation = spark.sql(f"""                    
                                    select distinct Agreement_Number
                                    from `p-cdh`.landed.profits_agreement_relation
                                    where BusinessDate = '{Businessdate}'
                                """)

agreement_relation.createOrReplaceTempView("agreement_relation")
# display(agreement_relation)

In [0]:
customer = spark.sql(f"""                    
                        select distinct CDI, cust_id
                        from `p-cdh`.landed.profits_customer
                        where BusinessDate = '{Businessdate}'
                    """)

customer.createOrReplaceTempView("customer")
# display(customer)

In [0]:
customer_address = spark.sql(f"""                    
                        select distinct cust_id
                        from `p-cdh`.landed.profits_customer_address
                        where BusinessDate = '{Businessdate}'
                    """)

customer_address.createOrReplaceTempView("customer_address")
# display(customer_address)

In [0]:
customer_category = spark.sql(f"""                    
                                select distinct cust_id
                                from `p-cdh`.landed.profits_customer_category
                                where BusinessDate = '{Businessdate}'
                            """)

customer_category.createOrReplaceTempView("customer_category")
# display(customer_category)

In [0]:
customer_relation = spark.sql(f"""                    
                                select distinct SEC_CUST_ID
                                from `p-cdh`.landed.profits_customer_relation
                                where BusinessDate = '{Businessdate}'
                            """)

customer_relation.createOrReplaceTempView("customer_relation")
# display(customer_relation)

In [0]:
DepositAccount = spark.sql(f"""                    
                                select distinct ACCOUNT_NUMBER,DEP_OPEN_UNIT
                                from `p-cdh`.landed.profits_DepositAccount
                                where BusinessDate = '{Businessdate}'
                            """)

DepositAccount.createOrReplaceTempView("DepositAccount")
# display(DepositAccount)

In [0]:
Group_member = spark.sql(f"""                    
                            select distinct CDI, AB_GROUP
                            from `p-cdh`.landed.profits_group_member
                            where BusinessDate = '{Businessdate}'
                        """)

Group_member.createOrReplaceTempView("Group_member")
# display(Group_member)

In [0]:
Loan_account_extrait = spark.sql(f"""                    
                            select distinct ACC_UNIT, ACC_SN, ACC_TYPE
                            from `p-cdh`.landed.profits_Loan_account_extrait
                            where BusinessDate = '{Businessdate}'
                        """)

Loan_account_extrait.createOrReplaceTempView("Loan_account_extrait")
# display(Loan_account_extrait)

## Check Tables

### Accounts

#### Check in loan_details

##### Check the number of Accounts per Prefix

In [0]:
NumberOfAccounts = spark.sql(f"""                    
                                select distinct port.Prefix, count(*)
                                from loan_details ld
                                inner join Portfolio port
                                on port.Prefix = left(ld.account_number,3)
                                group by all
                            """)

NumberOfAccounts.createOrReplaceTempView("NumberOfAccounts")
display(NumberOfAccounts)

##### Check that the Accounts have the right unit_code depending on Prefix

In [0]:
RightUnitCode = spark.sql(f"""                    
                              select distinct 
                                    port.Prefix
                                  , unit_code_profits AS Portfolio_unit_code
                                  , prft_acc_unit     AS loan_details_unit_code
                              from loan_details ld
                              inner join Portfolio port
                              on port.Prefix = left(ld.account_number,3)
                          """)

RightUnitCode.createOrReplaceTempView("RightUnitCode")
display(RightUnitCode)

##### Check that the Accounts have the right Prefix depending on unit_code

In [0]:
RightPrefix = spark.sql(f"""                    
                            select distinct 
                                    port.unit_code_profits      AS unit_code
                                  , port.Prefix                 AS Portfolio_Prefix
                                  , left(ld.account_number,3)   AS loan_details_Prefix
                            from loan_details ld
                            inner join Portfolio port
                            on port.unit_code_profits = ld.prft_acc_unit
                        """)

RightPrefix.createOrReplaceTempView("RightPrefix")
display(RightPrefix)

#### Check in loan_balance

In [0]:
Accounts_loan_balance = spark.sql(f"""                    
                                    select distinct port.Prefix, count(*)
                                    from loan_balance lb
                                    inner join Portfolio port
                                    on port.Prefix = left(lb.account_number,3)
                                    GROUP BY all
                                """)

Accounts_loan_balance.createOrReplaceTempView("Accounts_loan_balance")
display(Accounts_loan_balance)

#### Check in loan_relation

In [0]:
Accounts_loan_relation = spark.sql(f"""                    
                                        select distinct port.Prefix, count(*)
                                        from loan_relation lr
                                        inner join Portfolio port
                                        on port.Prefix = left(lr.FK_ACCOUNT,3)
                                        GROUP BY all
                                    """)

Accounts_loan_relation.createOrReplaceTempView("Accounts_loan_relation")
display(Accounts_loan_relation)

#### Check in loan_request

In [0]:
Accounts_loan_request = spark.sql(f"""                    
                                        select distinct port.Prefix, count(distinct lr.ACCOUNT_NUMBER)
                                        from loan_request lr
                                        inner join loan_details ld
                                        on lr.ACCOUNT_NUMBER = ld.account_number
                                        and lr.unit_code = ld.prft_acc_unit
                                        inner join Portfolio port
                                        on port.Prefix = left(lr.ACCOUNT_NUMBER,3)
                                        GROUP BY all
                                    """)

Accounts_loan_request.createOrReplaceTempView("Accounts_loan_request")
display(Accounts_loan_request)

#### Check in TUN

In [0]:
Accounts_TUN = spark.sql(f"""                    
                            select distinct port.Prefix, count(*)
                            from TUN
                            inner join Portfolio port
                            on port.Prefix = left(TUN.ACCOUNT_NUMBER,3)
                            GROUP BY all
                        """)

Accounts_TUN.createOrReplaceTempView("Accounts_TUN")
display(Accounts_TUN)

#### Check in write_off_account

In [0]:
Accounts_write_off_account = spark.sql(f"""                    
                            select distinct port.Prefix, count(*)
                            from write_off_account woa
                            inner join Portfolio port
                            on port.Prefix = left(woa.ACCOUNT_NUMBER,3)
                            GROUP BY all
                        """)

Accounts_write_off_account.createOrReplaceTempView("Accounts_write_off_account")
display(Accounts_write_off_account)

### Agreements

#### Check in loan_details

##### Check the number of Agreements per Prefix

In [0]:
NumberOfAgreements = spark.sql(f"""                    
                                    select distinct port.Prefix, count(*)
                                    from loan_details ld
                                    inner join Portfolio port
                                    on port.Prefix = left(ld.Agreement_Number,3)
                                    group by all
                                """)

NumberOfAgreements.createOrReplaceTempView("NumberOfAgreements")
display(NumberOfAgreements)

##### Check that all Accounts has an Agreement

In [0]:
Account_Agreement = spark.sql(f"""                    
                                select distinct 
                                        port.Prefix
                                      , left(ACCOUNT_NUMBER,3)
                                      , left(AGREEMENT_NUMBER,3)
                                from loan_details ld
                                inner join Portfolio port
                                on port.Prefix = left(ld.account_number,3)
                            """)

Account_Agreement.createOrReplaceTempView("Account_Agreement")
display(Account_Agreement)

#### Check in agreement

##### Check the number of Agreement per Prefix

In [0]:
NumberOfAgreements = spark.sql(f"""                    
                                select distinct port.Prefix, count(*)
                                from agreement agr
                                inner join Portfolio port
                                on port.Prefix = left(agr.agreement_number,3)
                                group by all
                            """)

NumberOfAgreements.createOrReplaceTempView("NumberOfAgreements")
display(NumberOfAgreements)

##### Check that the Agreement have the right unit_code depending on Prefix

In [0]:
RightUnitCode = spark.sql(f"""                    
                              select distinct 
                                    port.Prefix
                                  , unit_code_profits AS Portfolio_unit_code
                                  , unit_code         AS agreement_unit_code
                              from agreement agr
                              inner join Portfolio port
                              on port.Prefix = left(agr.Agreement_Number,3)
                          """)

RightUnitCode.createOrReplaceTempView("RightUnitCode")
display(RightUnitCode)

#### Check in agreement_relation

In [0]:
NumberOfAgreement_Relation = spark.sql(f"""                    
                                        select distinct port.Prefix, count(distinct ar.Agreement_Number)
                                        from agreement_relation ar
                                        inner join Portfolio port
                                        on port.Prefix = left(ar.agreement_number,3)
                                        group by all
                                    """)

NumberOfAgreement_Relation.createOrReplaceTempView("NumberOfAgreement_Relation")
display(NumberOfAgreement_Relation)

### Customers

#### Check in customer

In [0]:
NumberOfCustomer = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct CDI)
                                from customer C
                                inner join Portfolio port
                                on port.Prefix = left(C.CDI,3)
                                group by all
                            """)

NumberOfCustomer.createOrReplaceTempView("NumberOfCustomer")
display(NumberOfCustomer)

#### Check in customer_address

In [0]:
NumberOfCustomer_Address = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct CDI)
                                from customer_address cust_addr
                                inner join customer cust
                                on cust_addr.cust_id = cust.cust_id 
                                inner join Portfolio port
                                on port.Prefix = left(cust.CDI,3)
                                group by all
                            """)

NumberOfCustomer_Address.createOrReplaceTempView("NumberOfCustomer_Address")
display(NumberOfCustomer_Address)

#### Check in customer_category

In [0]:
NumberOfCustomer_Category = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct CDI)
                                from customer_category cust_categ
                                inner join customer cust
                                on cust_categ.cust_id = cust.cust_id 
                                inner join Portfolio port
                                on port.Prefix = left(cust.CDI,3)
                                group by all
                            """)

NumberOfCustomer_Category.createOrReplaceTempView("NumberOfCustomer_Category")
display(NumberOfCustomer_Category)

#### Check in customer_relation

In [0]:
NumberOfCustomer_Relation = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct CDI)
                                from customer_relation cust_rel
                                inner join customer cust
                                on cust_rel.SEC_CUST_ID = cust.cust_id 
                                inner join Portfolio port
                                on port.Prefix = left(cust.CDI,3)
                                group by all
                            """)

NumberOfCustomer_Relation.createOrReplaceTempView("NumberOfCustomer_Relation")
display(NumberOfCustomer_Relation)

### ---------------------------------

### Other

#### Check in DepositAccount

In [0]:
DEP_OPEN_UNIT_per_Prefix = spark.sql(f"""                    
                                        select distinct port.Prefix, DEP_OPEN_UNIT
                                        from DepositAccount
                                        inner join Portfolio port
                                        on port.Prefix = left(ACCOUNT_NUMBER,3)
                                    """)

DEP_OPEN_UNIT_per_Prefix.createOrReplaceTempView("DEP_OPEN_UNIT_per_Prefix")
display(DEP_OPEN_UNIT_per_Prefix)

#### Check in Group_member

In [0]:
Group_member_per_CDI = spark.sql(f"""                    
                                    select distinct port.Prefix, count(distinct CDI)
                                    from Group_member
                                    inner join Portfolio port
                                    on port.Prefix = left(CDI,3)
                                    where coalesce(AB_GROUP, '') <> ''
                                    group by all
                                """)

Group_member_per_CDI.createOrReplaceTempView("Group_member_per_CDI")
display(Group_member_per_CDI)

#### Check in Loan_account_extrait

In [0]:
NumberOfAccounts = spark.sql(f"""                    
                                select distinct port.Prefix, count(distinct ld.account_number)
                                from Loan_account_extrait ext
                                inner join loan_details ld
                                ON ld.prft_acc_unit = ext.ACC_UNIT 
                                AND ld.prft_acc_sn = ext.ACC_SN 
                                AND ld.prft_acc_type = ext.ACC_TYPE
                                inner join Portfolio port
                                on port.Prefix = left(ld.account_number,3)
                                group by all
                            """)

NumberOfAccounts.createOrReplaceTempView("NumberOfAccounts")
display(NumberOfAccounts)